# Choosing a model, an endpoint, and an API on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

A live capability survey across every family on `bedrock-mantle`. Rather than
trusting a table someone wrote months ago, this notebook **probes the endpoint**
and builds the table from what the API actually does today.

Use it to answer: *which model, which API, which path, which parameters?*

## What this notebook produces
- The full model inventory and its Region footprint
- A per-family API matrix (Responses / Chat Completions / Messages)
- A per-model parameter matrix (`temperature`, `top_p`, `service_tier`, …)
- A reusable `capabilities.py` you can drop into your own project

## Self-contained, but see also
- **Auth, the three URL paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- Family deep-dives: `../01-openai-gpt/` … `../12-writer-palmyra/`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import concurrent.futures as cf
import json
import re
import sys
from collections import defaultdict

sys.path.insert(0, "../_shared")
from bedrock import err, list_models, ok, post, runtime_post

REGION = "us-east-1"  # the widest inventory
REGIONS = ("us-east-1", "us-east-2", "us-west-2", "eu-central-1")
print("survey region:", REGION)

survey region: us-east-1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-4-7",
    "anthropic.claude-opus-4-8",
    "anthropic.claude-opus-5",
    "anthropic.claude-sonnet-5",
    "deepseek.v3.2",
    "google.gemma-4-31b",
    "minimax.minimax-m2.5",
    "mistral.mistral-large-3-675b-instruct",
    "moonshotai.kimi-k2.5",
    "nvidia.nemotron-super-3-120b",
    "openai.gpt-5.5",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "openai.gpt-oss-safeguard-20b",
    "qwen.qwen3-32b",
    "writer.palmyra-vision-7b",
    "xai.grok-4.3",
    "zai.glm-5",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


anthropic.claude-haiku-4-5             us.anthropic.claude-haiku-4-5-20251001-v1:0 mantle, runtime


anthropic.claude-opus-4-7              us.anthropic.claude-opus-4-7             mantle, runtime


anthropic.claude-opus-4-8              us.anthropic.claude-opus-4-8             mantle, runtime


anthropic.claude-opus-5                us.anthropic.claude-opus-5               mantle, runtime


...

qwen.qwen3-32b                         qwen.qwen3-32b-v1:0                      mantle, runtime


writer.palmyra-vision-7b               writer.palmyra-vision-7b                 mantle, runtime


xai.grok-4.3                           -- not on runtime --                     mantle


zai.glm-5                              zai.glm-5                                mantle, runtime



=> 16/19 of these are on bedrock-runtime; 8 under a different id.
   bedrock-mantle only: ['google.gemma-4-31b', 'openai.gpt-5.5', 'xai.grok-4.3']
   For those, this notebook's endpoint is the only one that serves them.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. Inventory and Region footprint

In [3]:
inventory = {}
for region in REGIONS:
    try:
        inventory[region] = sorted(list_models(region))
    except (RuntimeError, OSError) as exc:
        inventory[region] = []
        print(f"{region}: {type(exc).__name__}")

for region in REGIONS:
    print(f"{region:14} {len(inventory[region]):3} models")

models = inventory[REGION]
families = defaultdict(list)
for mid in models:
    families[mid.split(".")[0]].append(mid)

print(f"\n{len(models)} models across {len(families)} families in {REGION}")

us-east-1       55 models
us-east-2       50 models
us-west-2       48 models
eu-central-1    33 models

55 models across 12 families in us-east-1


In [4]:
print(f"{'family':14} {'n':>3}  models")
print("-" * 100)
for family in sorted(families):
    ids = families[family]
    print(
        f"{family:14} {len(ids):>3}  {', '.join(i.split('.', 1)[1] for i in ids)[:76]}"
    )

family           n  models
----------------------------------------------------------------------------------------------------
anthropic        6  claude-fable-5, claude-haiku-4-5, claude-opus-4-7, claude-opus-4-8, claude-o
deepseek         2  v3.1, v3.2
google           6  gemma-3-12b-it, gemma-3-27b-it, gemma-3-4b-it, gemma-4-26b-a4b, gemma-4-31b,
minimax          3  minimax-m2, minimax-m2.1, minimax-m2.5
mistral          8  devstral-2-123b, magistral-small-2509, ministral-3-14b-instruct, ministral-3
moonshotai       2  kimi-k2-thinking, kimi-k2.5
nvidia           4  nemotron-nano-12b-v2, nemotron-nano-3-30b, nemotron-nano-9b-v2, nemotron-sup
openai          11  gpt-5.4, gpt-5.4-2026-03-05, gpt-5.5, gpt-5.5-2026-04-23, gpt-5.6-luna, gpt-
qwen             7  qwen3-235b-a22b-2507, qwen3-32b, qwen3-coder-30b-a3b-instruct, qwen3-coder-4
writer           1  palmyra-vision-7b
xai              1  grok-4.3
zai              4  glm-4.6, glm-4.7, glm-4.7-flash, glm-5


In [5]:
# Which models are missing where? This is the deployment-planning view.
everywhere = set(inventory[REGIONS[0]])
for region in REGIONS[1:]:
    everywhere &= set(inventory[region])
print(f"available in ALL four Regions: {len(everywhere)}")

only_one = [m for m in models if sum(1 for r in REGIONS if m in inventory[r]) == 1]
print(f"available in only ONE Region : {len(only_one)}")
for mid in sorted(only_one):
    where = [r for r in REGIONS if mid in inventory[r]]
    print(f"   {mid:44} {where[0]}")

available in ALL four Regions: 33
available in only ONE Region : 5
   anthropic.claude-fable-5                     us-east-1
   anthropic.claude-opus-4-7                    us-east-1
   anthropic.claude-opus-4-8                    us-east-1
   anthropic.claude-opus-5                      us-east-1
   anthropic.claude-sonnet-5                    us-east-1


## 2. Path resolution — per endpoint, not just per model

Prefixes depend on the model family **and** on the endpoint, and the split falls in
a different place on each:

| | `bedrock-mantle` | `bedrock-runtime` |
|---|---|---|
| `/openai/v1` | `google.gemma-4*`, `openai.gpt-5*`, `xai.*` | **every** OpenAI-compatible model |
| `/v1` | everything else non-Anthropic | **does not exist** |
| `/anthropic/v1` | `anthropic.*` | `anthropic.*` |

So `openai.gpt-oss-120b` is `/v1` on mantle and its runtime twin
`openai.gpt-oss-120b-1:0` is `/openai/v1`. A resolver that takes only a model ID can
be right about one endpoint at a time — this collection shipped exactly that until
`bedrock-runtime` grew the OpenAI-compatible paths in August 2026.

Copy the version below, with the `endpoint` argument.

In [6]:
import re


PROFILE_RE = re.compile(r"^(us|eu|apac|global|in)\.")


def api_prefix(model_id: str, endpoint: str = "mantle") -> str:
    """Which URL prefix serves this model's inference APIs, on this endpoint?"""
    # Strip a geo/global inference-profile prefix first: "us.anthropic.claude-opus-5"
    # does not start with "anthropic.", and a naive check routes it to /openai/v1.
    bare = PROFILE_RE.sub("", model_id)
    if bare.startswith("anthropic."):
        return "/anthropic/v1"
    if endpoint == "runtime":
        return "/openai/v1"
    if bare.startswith(("google.gemma-4", "openai.gpt-5", "xai.")):
        return "/openai/v1"
    return "/v1"


by_prefix = defaultdict(list)
for mid in models:
    by_prefix[api_prefix(mid)].append(mid)

print("on bedrock-mantle:")
for prefix in sorted(by_prefix):
    ids = by_prefix[prefix]
    fams = sorted({i.split(".")[0] for i in ids})
    print(f"  {prefix:16} {len(ids):3} models | families: {', '.join(fams)}")

# The same models, addressed on bedrock-runtime. Note that the /v1 bucket empties.
runtime_ids = [r for m in models if (r := runtime_id_for(m, REGION)) is not None]
runtime_by_prefix = defaultdict(list)
for rid in runtime_ids:
    runtime_by_prefix[api_prefix(rid, "runtime")].append(rid)

print(f"\non bedrock-runtime ({len(runtime_ids)} of {len(models)} are there):")
for prefix in sorted(runtime_by_prefix):
    ids = runtime_by_prefix[prefix]
    fams = sorted({PROFILE_RE.sub("", i).split(".")[0] for i in ids})
    print(f"  {prefix:16} {len(ids):3} models | families: {', '.join(fams)}")

moved = [
    m for m in models
    if (r := runtime_id_for(m, REGION)) is not None
    and api_prefix(m) != api_prefix(r, "runtime")
]
print(f"\n=> {len(moved)} model(s) change PATH between endpoints, e.g. "
      f"{moved[:3]}")
print(f"=> bedrock-runtime serves no /v1 inference path at all: "
      f"{'/v1' not in runtime_by_prefix}")

on bedrock-mantle:
  /anthropic/v1      6 models | families: anthropic
  /openai/v1        11 models | families: google, openai, xai
  /v1               38 models | families: deepseek, google, minimax, mistral, moonshotai, nvidia, openai, qwen, writer, zai

on bedrock-runtime (43 of 55 are there):
  /anthropic/v1      6 models | families: anthropic
  /openai/v1        37 models | families: deepseek, google, minimax, mistral, moonshot, moonshotai, nvidia, openai, qwen, writer, zai

=> 34 model(s) change PATH between endpoints, e.g. ['deepseek.v3.2', 'google.gemma-3-12b-it', 'google.gemma-3-27b-it']
=> bedrock-runtime serves no /v1 inference path at all: True


Note the split *inside* the OpenAI family: `gpt-5.*` uses `/openai/v1` while
`gpt-oss*` uses the bare `/v1`. Provider name alone is not enough.

In [7]:
for mid in sorted(families["openai"]):
    print(f"  {mid:34} -> {api_prefix(mid)}")

  openai.gpt-5.4                     -> /openai/v1
  openai.gpt-5.4-2026-03-05          -> /openai/v1
  openai.gpt-5.5                     -> /openai/v1
  openai.gpt-5.5-2026-04-23          -> /openai/v1
  openai.gpt-5.6-luna                -> /openai/v1
  openai.gpt-5.6-sol                 -> /openai/v1
  openai.gpt-5.6-terra               -> /openai/v1
  openai.gpt-oss-120b                -> /v1
  openai.gpt-oss-20b                 -> /v1
  openai.gpt-oss-safeguard-120b      -> /v1
  openai.gpt-oss-safeguard-20b       -> /v1


## 3. Probe the API surface per family

One representative per family, three APIs each. Run in parallel to keep it quick.

In [8]:
REPRESENTATIVES = [
    "google.gemma-4-31b",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "anthropic.claude-haiku-4-5",
    "xai.grok-4.3",
    "qwen.qwen3-32b",
    "deepseek.v3.2",
    "zai.glm-5",
    "minimax.minimax-m2.5",
    "moonshotai.kimi-k2.5",
    "mistral.mistral-large-3-675b-instruct",
    "nvidia.nemotron-super-3-120b",
    "writer.palmyra-vision-7b",
]
REPRESENTATIVES = [m for m in REPRESENTATIVES if m in models]
AV = {"anthropic-version": "2023-06-01"}


def probe_apis(model_id: str) -> dict:
    """Status code per API. Single attempt + short timeout: a wrong path can stall."""
    prefix = api_prefix(model_id)
    out = {"model": model_id, "prefix": prefix}

    if prefix == "/anthropic/v1":
        code, _ = post(
            f"{prefix}/messages",
            {
                "model": model_id,
                "max_tokens": 16,
                "messages": [{"role": "user", "content": "Hi"}],
            },
            region=REGION,
            headers=AV,
            attempts=1,
            timeout=45,
        )
        out["messages"] = code
        out["responses"] = out["chat"] = "-"
        out["chat_field"] = "-"
        return out

    # Try BOTH budget field names before concluding an API is missing. gpt-5.6
    # serves Chat Completions but refuses `max_tokens`, and reading that 400 as
    # "no Chat Completions" is how a false claim reached three other notebooks.
    code, _ = post(
        f"{prefix}/responses",
        {"model": model_id, "input": "Hi", "max_output_tokens": 16},
        region=REGION,
        attempts=1,
        timeout=45,
    )
    out["responses"] = code
    for field in ("max_tokens", "max_completion_tokens"):
        code, _ = post(
            f"{prefix}/chat/completions",
            {
                "model": model_id,
                "messages": [{"role": "user", "content": "Hi"}],
                field: 16,
            },
            region=REGION,
            attempts=1,
            timeout=45,
        )
        if code == 200:
            out["chat_field"] = field
            break
    out["chat"] = code
    out.setdefault("chat_field", "-")
    out["messages"] = "-"
    return out


with cf.ThreadPoolExecutor(max_workers=5) as pool:
    api_results = list(pool.map(probe_apis, REPRESENTATIVES))

print(f"{'model':40} {'prefix':16} {'Resp':>6} {'Chat':>6} {'Msg':>6} {'CC budget':>22}")
print("-" * 104)
for row in api_results:
    print(
        f"{row['model']:40} {row['prefix']:16} {str(row['responses']):>6} "
        f"{str(row['chat']):>6} {str(row['messages']):>6} "
        f"{str(row.get('chat_field', '-')):>22}"
    )

model                                    prefix             Resp   Chat    Msg              CC budget
--------------------------------------------------------------------------------------------------------
google.gemma-4-31b                       /openai/v1          200    200      -             max_tokens
openai.gpt-5.6-sol                       /openai/v1          200    200      -  max_completion_tokens
openai.gpt-oss-120b                      /v1                 200    200      -             max_tokens
anthropic.claude-haiku-4-5               /anthropic/v1         -      -    200                      -
xai.grok-4.3                             /openai/v1          200    200      -             max_tokens
qwen.qwen3-32b                           /v1                 400    200      -             max_tokens
deepseek.v3.2                            /v1                 400    200      -             max_tokens
zai.glm-5                                /v1                 400    200      - 

`200` means available; `400` means "this model does not serve that API"; `-1`
means the request **stalled** rather than erroring — which is why every probe
above sets a timeout.

In [9]:
summary = {"responses": [], "chat": [], "messages": []}
for row in api_results:
    for key in summary:
        if row[key] == 200:
            summary[key].append(row["model"].split(".")[0])
print("Responses API       :", sorted(set(summary["responses"])))
print("Chat Completions    :", sorted(set(summary["chat"])))
print("Anthropic Messages  :", sorted(set(summary["messages"])))

Responses API       : ['google', 'openai', 'xai']
Chat Completions    : ['deepseek', 'google', 'minimax', 'mistral', 'moonshotai', 'nvidia', 'openai', 'qwen', 'writer', 'xai', 'zai']
Anthropic Messages  : ['anthropic']


## 4. Parameter compatibility

Sampling parameters are the most common cross-family break. There is **no single
config that works everywhere** — this table proves it.

In [10]:
def probe_params(model_id: str) -> dict:
    prefix = api_prefix(model_id)
    out = {"model": model_id}

    if prefix == "/anthropic/v1":
        base = {
            "model": model_id,
            "max_tokens": 16,
            "messages": [{"role": "user", "content": "Hi"}],
        }
        for label, extra in (
            ("temperature", {"temperature": 0.5}),
            ("top_p", {"top_p": 0.9}),
        ):
            code, _ = post(
                f"{prefix}/messages",
                {**base, **extra},
                region=REGION,
                headers=AV,
                attempts=1,
                timeout=45,
            )
            out[label] = "ok" if code == 200 else str(code)
        out["service_tier"] = "-"
        return out

    # Use whichever inference API this model actually serves.
    code, _ = post(
        f"{prefix}/responses",
        {"model": model_id, "input": "Hi", "max_output_tokens": 16},
        region=REGION,
        attempts=1,
        timeout=45,
    )
    if code == 200:
        path, base = f"{prefix}/responses", {
            "model": model_id,
            "input": "Hi",
            "max_output_tokens": 16,
        }
    else:
        path, base = (
            f"{prefix}/chat/completions",
            {
                "model": model_id,
                "messages": [{"role": "user", "content": "Hi"}],
                "max_tokens": 16,
            },
        )

    for label, extra in (
        ("temperature", {"temperature": 0.5}),
        ("top_p", {"top_p": 0.9}),
        ("service_tier", {"service_tier": "flex"}),
    ):
        code, _ = post(path, {**base, **extra}, region=REGION, attempts=1, timeout=45)
        out[label] = "ok" if code == 200 else str(code)
    return out


with cf.ThreadPoolExecutor(max_workers=5) as pool:
    param_results = list(pool.map(probe_params, REPRESENTATIVES))

print(f"{'model':40} {'temperature':>12} {'top_p':>8} {'flex tier':>11}")
print("-" * 76)
for row in param_results:
    print(
        f"{row['model']:40} {row['temperature']:>12} {row['top_p']:>8} "
        f"{row['service_tier']:>11}"
    )

model                                     temperature    top_p   flex tier
----------------------------------------------------------------------------
google.gemma-4-31b                                 ok       ok          ok
openai.gpt-5.6-sol                                400      400         400
openai.gpt-oss-120b                                ok       ok          ok
anthropic.claude-haiku-4-5                         ok       ok           -
xai.grok-4.3                                       ok       ok          ok
qwen.qwen3-32b                                     ok       ok          ok
deepseek.v3.2                                      ok       ok          ok
zai.glm-5                                          ok       ok          ok
minimax.minimax-m2.5                               ok       ok          ok
moonshotai.kimi-k2.5                               ok       ok          ok
mistral.mistral-large-3-675b-instruct              ok       ok          ok
nvidia.nemotron-super-3

Read the table, not this paragraph — the paragraph is the part that goes stale.
When this notebook was first written it claimed Gemma 4 took `temperature` and
refused `top_p`, directly beneath a table showing `temperature` refused. Both the
table and the prose have since been wrong in different directions.

What the survey reliably shows:

- **There is no universal sampling config.** The GPT-5.5 and GPT-5.6 families
  accept `temperature` only at its default `1.0` and refuse `top_p`; newer Claude
  models refuse both as *deprecated*; `haiku-4-5` accepts either but not both in
  one request. Everything else is permissive.
- **`gpt-5.x` and Claude reject `flex`/`priority`.** For Claude, `service_tier` is
  not a Messages parameter at all, so `n/a` rather than a refusal.
- **A `-1` means the request stalled**, not that anything was rejected. That is why
  every probe here sets a timeout.

Resolve sampling and tier per model, and re-probe: three of these rows have
changed since the notebook was written.

## 5. Structured-output support

In [11]:
SCHEMA = {
    "type": "object",
    "properties": {"answer": {"type": "string"}},
    "required": ["answer"],
    "additionalProperties": False,
}


ASK = [{"role": "user", "content": "Answer 'hi'."}]
PROBE = {"region": REGION, "attempts": 1, "timeout": 60}


def _claude_forced_tool(model_id: str, prefix: str) -> int:
    """Claude: output_config.format is rejected on mantle; forced tools work."""
    code, _ = post(
        f"{prefix}/messages",
        {
            "model": model_id,
            "max_tokens": 300,
            "messages": ASK,
            "tools": [
                {"name": "emit", "description": "Return.", "input_schema": SCHEMA}
            ],
            "tool_choice": {"type": "tool", "name": "emit"},
        },
        headers=AV,
        **PROBE,
    )
    return code


def _native_schema(model_id: str, prefix: str) -> str:
    """Try Responses text.format, then fall back to Chat Completions."""
    code, _ = post(
        f"{prefix}/responses",
        {
            "model": model_id,
            "input": "Answer 'hi'.",
            "max_output_tokens": 300,
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "s",
                    "schema": SCHEMA,
                    "strict": True,
                }
            },
        },
        **PROBE,
    )
    if code in (200, 400):
        return "ok" if code == 200 else "400"

    # No Responses API for this model; try Chat Completions response_format.
    code, _ = post(
        f"{prefix}/chat/completions",
        {
            "model": model_id,
            "messages": ASK,
            "max_tokens": 300,
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": "s", "strict": True, "schema": SCHEMA},
            },
        },
        **PROBE,
    )
    return "ok (CC)" if code == 200 else str(code)


def _forced_tool(model_id: str, prefix: str) -> str:
    """Forced function call — the portable structured-output route."""
    code, _ = post(
        f"{prefix}/chat/completions",
        {
            "model": model_id,
            "messages": ASK,
            "max_tokens": 300,
            "tools": [
                {
                    "type": "function",
                    "function": {
                        "name": "emit",
                        "description": "Return.",
                        "parameters": SCHEMA,
                    },
                }
            ],
            "tool_choice": {"type": "function", "function": {"name": "emit"}},
        },
        **PROBE,
    )
    return "ok" if code == 200 else str(code)

One dispatcher over the three probes above.

In [12]:
def probe_structured(model_id: str) -> dict:
    """Which structured-output mechanisms does this model actually accept?"""
    prefix = api_prefix(model_id)
    if prefix == "/anthropic/v1":
        code = _claude_forced_tool(model_id, prefix)
        return {
            "model": model_id,
            "native_schema": "n/a",
            "forced_tool": "ok" if code == 200 else str(code),
        }
    return {
        "model": model_id,
        "native_schema": _native_schema(model_id, prefix),
        "forced_tool": _forced_tool(model_id, prefix),
    }


with cf.ThreadPoolExecutor(max_workers=4) as pool:
    struct_results = list(pool.map(probe_structured, REPRESENTATIVES))

print(f"{'model':40} {'native schema':>15} {'forced tool':>13}")
print("-" * 72)
for row in struct_results:
    print(f"{row['model']:40} {row['native_schema']:>15} {row['forced_tool']:>13}")

model                                      native schema   forced tool
------------------------------------------------------------------------
google.gemma-4-31b                                    ok            ok
openai.gpt-5.6-sol                                    ok           400
openai.gpt-oss-120b                              ok (CC)            ok
anthropic.claude-haiku-4-5                           n/a            ok
xai.grok-4.3                                          ok            ok
qwen.qwen3-32b                                       400            ok
deepseek.v3.2                                        400            ok
zai.glm-5                                            400            ok
minimax.minimax-m2.5                                 400            ok
moonshotai.kimi-k2.5                                 400            ok
mistral.mistral-large-3-675b-instruct                400            ok
nvidia.nemotron-super-3-120b                         400            ok
writ

**Caveat worth remembering:** "accepted" is not "reliable". A request can return
200 while the model ignores the constraint (gpt-oss does this with a named
`tool_choice`) or appends characters after a valid JSON object (Gemma 4 does this
roughly half the time). Always validate the payload you get back.

## 6. A reusable capabilities resolver

Everything above, distilled into something you can paste into a project.

In [13]:
CAPABILITIES_PY = '''
"""Resolve Bedrock request details per model and endpoint. From live probes.

Read the caveat before you rely on this. The constants below are a *snapshot*:
which models refuse which parameters has changed twice during this collection's
life. A resolver built on a static table is right until it is not, and it fails
closed in the worst way -- a 400 in production on a model you never re-tested.

The durable design is the one in `02-migrating-from-openai.ipynb` section 7: send
the request, and when the service returns a 400 that *names* a parameter, drop that
parameter and retry. Bedrock is consistent about naming it. Use this module for
routing (which path, which API, which budget field) and let error handling deal
with sampling.

Every routing function takes `endpoint="mantle"` or `endpoint="runtime"`, because
the path, the API surface and the model ID all differ between the two. It does NOT
translate model IDs -- ask the service, via `runtime_id_for()` in
`_shared/bedrock.py`, rather than encoding a mapping that will age.
"""
import re

OPENAI_PREFIX_FAMILIES = ("google.gemma-4", "openai.gpt-5", "xai.")

# A geo/global inference-profile prefix is not part of the family name, and
# bedrock-runtime requires one for several families. Strip it before matching.
PROFILE_PREFIX = re.compile(r"^(us|eu|apac|global|in)[.]")

# Families served by Chat Completions only -- the Responses API 400s for these.
# Note gpt-oss-safeguard is here while base gpt-oss is not: the provider prefix
# is not enough to decide.
CHAT_ONLY = ("qwen.", "deepseek.", "zai.", "minimax.", "moonshotai.", "mistral.",
             "nvidia.", "writer.", "openai.gpt-oss-safeguard", "google.gemma-3")

# Chat Completions wants max_completion_tokens rather than max_tokens for these.
COMPLETION_TOKENS_FAMILIES = ("openai.gpt-5.6",)

# Models that reject `temperature` and `top_p` outright ("deprecated").
NO_SAMPLING = ("anthropic.claude-opus-5", "anthropic.claude-sonnet-5",
               "anthropic.claude-opus-4-8", "anthropic.claude-opus-4-7")
# Models that accept EITHER temperature or top_p but not both in one request.
ONE_SAMPLING_PARAM = ("anthropic.claude-haiku-4-5",)
# Models that accept `temperature` ONLY at its default 1.0, and reject `top_p`.
TEMPERATURE_DEFAULT_ONLY = ("openai.gpt-5.5", "openai.gpt-5.6")
# Models that accept flex/priority service tiers. Not a Messages parameter at all.
TIERED = ("openai.gpt-oss", "google.gemma-", "xai.", "qwen.", "deepseek.",
          "zai.", "minimax.", "moonshotai.", "mistral.", "nvidia.", "writer.")

# On bedrock-runtime the Responses API reaches only these families; everything
# else OpenAI-compatible there is Chat Completions. Much narrower than on mantle.
RUNTIME_RESPONSES = ("openai.gpt-5.6", "xai.grok-4.6")

# Features that exist on exactly one endpoint. Check before you pick.
MANTLE_ONLY_FEATURES = ("server_side_tools", "web_search", "background",
                        "projects", "workspaces")
RUNTIME_ONLY_FEATURES = ("guardrails", "prompt_routing", "cross_region",
                         "provisioned_throughput", "batch")


def api_prefix(model_id, endpoint="mantle"):
    """Return the URL prefix that serves this model's inference APIs.

    `endpoint` is "mantle" or "runtime", and it changes the answer: runtime serves
    every OpenAI-compatible model on /openai/v1 and has no /v1 inference path.
    """
    bare = PROFILE_PREFIX.sub("", model_id)
    if bare.startswith("anthropic."):
        return "/anthropic/v1"
    if endpoint == "runtime":
        return "/openai/v1"
    if bare.startswith(OPENAI_PREFIX_FAMILIES):
        return "/openai/v1"
    return "/v1"


def surface(model_id, endpoint="mantle"):
    """Which API this model actually serves: messages / chat / responses.

    Narrower on bedrock-runtime, where only the GPT-5.6 and Grok 4.6 profiles
    serve Responses and only the newest Claude models serve Messages.
    """
    bare = PROFILE_PREFIX.sub("", model_id)
    if bare.startswith("anthropic."):
        return "messages"
    if endpoint == "runtime":
        return "responses" if bare.startswith(RUNTIME_RESPONSES) else "chat"
    if bare.startswith(CHAT_ONLY):
        return "chat"
    return "responses"


def inference_path(model_id, api="auto", endpoint="mantle"):
    """Full inference path, e.g. "/v1/chat/completions".

    `api="auto"` picks the surface the model actually serves. Defaulting to
    /responses for everything non-Anthropic -- which an earlier version of this
    module did -- 400s for every Chat-Completions-only family.
    """
    prefix = api_prefix(model_id, endpoint)
    chosen = surface(model_id, endpoint) if api == "auto" else api
    if chosen == "messages":
        return prefix + "/messages"
    if chosen == "chat":
        return prefix + "/chat/completions"
    return prefix + "/responses"


def token_limit_field(model_id, api="auto", endpoint="mantle"):
    """The budget parameter this model+API expects.

    Getting this wrong yields a 400 that reads exactly like "this API does not
    exist here", which is a trap worth avoiding by construction.
    """
    chosen = surface(model_id, endpoint) if api == "auto" else api
    if chosen == "responses":
        return "max_output_tokens"          # minimum 16
    if chosen == "chat" and PROFILE_PREFIX.sub("", model_id).startswith(
            COMPLETION_TOKENS_FAMILIES):
        return "max_completion_tokens"
    return "max_tokens"                     # Messages: required, no default


def sampling(model_id, temperature=None, top_p=None):
    """Drop or clamp parameters this model rejects, instead of earning a 400.

    Match on the model ID with any geo/global prefix removed. Matching the raw ID
    is a real bug and a quiet one: `us.anthropic.claude-opus-5` does not start with
    `anthropic.claude-opus-5`, so every rule below stops applying the moment you
    move to bedrock-runtime, and the 400 you get back names `temperature` on a
    model this table already knew rejects it. The verification cell below is what
    found that.
    """
    bare = PROFILE_PREFIX.sub("", model_id)
    if bare.startswith(NO_SAMPLING):
        return {}
    out = {}
    if temperature is not None:
        if bare.startswith(TEMPERATURE_DEFAULT_ONLY):
            if abs(float(temperature) - 1.0) < 1e-9:
                out["temperature"] = 1.0
            # else omit entirely rather than 400
        else:
            out["temperature"] = temperature
    if top_p is not None and not bare.startswith(TEMPERATURE_DEFAULT_ONLY):
        out["top_p"] = top_p
    # "`temperature` and `top_p` cannot both be specified for this model."
    if bare.startswith(ONE_SAMPLING_PARAM) and len(out) == 2:
        out.pop("top_p")
    return out


def service_tier(model_id, tier="default"):
    """Downgrade to 'default' where flex/priority are unsupported.

    Returns None for Messages, where service_tier is not a parameter at all.
    """
    if surface(model_id) == "messages":
        return None
    if tier == "default" or model_id.startswith(TIERED):
        return tier
    return "default"
'''

with open("capabilities.py", "w") as handle:
    handle.write(CAPABILITIES_PY)

sys.path.insert(0, ".")
import capabilities

print(f"{'model':32} {'path':30} {'budget field':22} sampling")
print("-" * 118)
for mid in (
    "google.gemma-4-31b",
    "xai.grok-4.3",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "openai.gpt-oss-safeguard-20b",
    "anthropic.claude-sonnet-5",
    "anthropic.claude-haiku-4-5",
    "qwen.qwen3-32b",
):
    print(
        f"{mid:32} {capabilities.inference_path(mid):30} "
        f"{capabilities.token_limit_field(mid):22} "
        f"{json.dumps(capabilities.sampling(mid, temperature=0.7, top_p=0.95))}"
    )

model                            path                           budget field           sampling
----------------------------------------------------------------------------------------------------------------------
google.gemma-4-31b               /openai/v1/responses           max_output_tokens      {"temperature": 0.7, "top_p": 0.95}
xai.grok-4.3                     /openai/v1/responses           max_output_tokens      {"temperature": 0.7, "top_p": 0.95}
openai.gpt-5.6-sol               /openai/v1/responses           max_output_tokens      {}
openai.gpt-oss-120b              /v1/responses                  max_output_tokens      {"temperature": 0.7, "top_p": 0.95}
openai.gpt-oss-safeguard-20b     /v1/chat/completions           max_tokens             {"temperature": 0.7, "top_p": 0.95}
anthropic.claude-sonnet-5        /anthropic/v1/messages         max_tokens             {}
anthropic.claude-haiku-4-5       /anthropic/v1/messages         max_tokens             {"temperature": 0.7}
qwen.

In [14]:
# Verify the resolver against the live endpoints. A resolver that is wrong is
# worse than none: it turns "read the docs" into "debug a 400 in production".
# So exercise it on BOTH endpoints and count the failures.
namespace: dict = {}
# CAPABILITIES_PY is the literal defined in the cell above, not input; this cell
# exists to prove that literal actually runs.
# nosemgrep: exec-detected
exec(CAPABILITIES_PY, namespace)  # nosec B102  # noqa: S102 - our own literal above

resolved_path = namespace["inference_path"]
resolved_field = namespace["token_limit_field"]
resolved_sampling = namespace["sampling"]

CHECK = [
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "anthropic.claude-opus-5",
    "qwen.qwen3-32b",
    "deepseek.v3.2",
    "zai.glm-5",
    "google.gemma-4-31b",
    "xai.grok-4.6",
]
AV = {"anthropic-version": "2023-06-01"}
print(f"{'model':26} {'endpoint':8} {'resolved path':30} {'field':22} result")
print("-" * 104)

failures = []
for model_id in CHECK:
    for endpoint in ("mantle", "runtime"):
        if endpoint == "mantle":
            send_id = model_id if model_id in models else None
        else:
            send_id = runtime_id_for(model_id, REGION)
        if send_id is None:
            print(f"{model_id:26} {endpoint:8} {'-- not on this endpoint --':30} "
                  f"{'-':22} skipped")
            continue

        path = resolved_path(send_id, endpoint=endpoint)
        field = resolved_field(send_id, endpoint=endpoint)
        body = {"model": send_id, field: 2000, **resolved_sampling(send_id, 0.5)}
        if path.endswith("/responses"):
            body["input"] = "Reply OK"
        else:
            body["messages"] = [{"role": "user", "content": "Reply OK"}]

        caller = post if endpoint == "mantle" else runtime_post
        code, data = caller(
            path, body, region=REGION,
            headers=AV if path.endswith("/messages") else None,
            attempts=1, timeout=120,
        )
        good = ok(code, data)
        if not good:
            failures.append((model_id, endpoint, code, err(data)[:60]))
        print(f"{model_id:26} {endpoint:8} {path:30} {field:22} "
              f"{'ok' if good else str(code) + ' ' + err(data)[:34]}")

print()
if failures:
    print(f"=> {len(failures)} combination(s) failed. The resolver is wrong for these,")
    print("   which is exactly the situation this cell exists to catch:")
    for model_id, endpoint, code, message in failures:
        print(f"     {model_id} on {endpoint}: {code} {message}")
else:
    print(f"=> every resolved request succeeded, on both endpoints, for the "
          f"{len(CHECK)} models above.")
    print("   That is the bar for pasting this into a project. Re-run it when you")
    print("   add a model, and treat a failure here as a resolver bug rather than")
    print("   a service problem.")

model                      endpoint resolved path                  field                  result
--------------------------------------------------------------------------------------------------------


openai.gpt-5.6-sol         mantle   /openai/v1/responses           max_output_tokens      ok


openai.gpt-5.6-sol         runtime  /openai/v1/responses           max_output_tokens      ok


openai.gpt-oss-120b        mantle   /v1/responses                  max_output_tokens      ok


openai.gpt-oss-120b        runtime  /openai/v1/chat/completions    max_tokens             ok


...

deepseek.v3.2              runtime  /openai/v1/chat/completions    max_tokens             ok


zai.glm-5                  mantle   /v1/chat/completions           max_tokens             ok


zai.glm-5                  runtime  /openai/v1/chat/completions    max_tokens             ok


google.gemma-4-31b         mantle   /openai/v1/responses           max_output_tokens      ok
google.gemma-4-31b         runtime  -- not on this endpoint --     -                      skipped
xai.grok-4.6               mantle   -- not on this endpoint --     -                      skipped


xai.grok-4.6               runtime  /openai/v1/responses           max_output_tokens      ok

=> every resolved request succeeded, on both endpoints, for the 8 models above.
   That is the bar for pasting this into a project. Re-run it when you
   add a model, and treat a failure here as a resolver bug rather than
   a service problem.


## 7. A decision guide

| If you need… | Use |
|---|---|
| Web Search grounding | `openai.gpt-5.*` (Responses) — the only family |
| Reasoning traces you can read | Responses API: gemma-4 (at `effort="high"`), gpt-5.x, gpt-oss. Grok's is **encrypted**. On Chat Completions several `/v1` families put it in the non-standard `message.reasoning` |
| Server-side tools (Lambda / Gateway) | Responses API models; built-ins on gpt-oss |
| Adaptive thinking + 1h prompt cache | `anthropic.claude-*` (Messages) |
| Explicit prompt-cache breakpoints | `openai.gpt-5.6-*` |
| Server-side conversation state | Responses API + `store=True` |
| Zero data retention | any model, `store=False` + retention mode `none` |
| EU data residency | check `eu-central-1` — no Anthropic, no gpt-5.x, no xAI |
| Vision | gemma-4, qwen3-vl, nemotron-nano-12b, palmyra-vision, Claude |
| Cheapest viable model | Ministral / Nemotron Nano / GLM Flash ladders |
| Fine-tuning | `gpt-oss-20b` or `qwen3-32b`, us-west-2 only |

## Gotchas this survey exposes

| Gotcha | Detail |
|---|---|
| Three path prefixes | And a split *within* the OpenAI family |
| Responses API coverage | Only a minority of families — and the gpt-oss **safeguard** variants lack it while base gpt-oss has it |
| Budget field name | Chat Completions wants `max_completion_tokens` on gpt-5.6. That 400 reads like a missing API |
| Claude is Messages-only | Both OpenAI-compatible APIs 400 |
| No universal sampling config | gpt-5.5/5.6 want `temperature=1.0` and refuse `top_p`; newer Claude refuses both; `haiku-4-5` takes either but not both |
| Tier support varies | gpt-5.x is `default`-only |
| Wrong path may stall | Always set a client-side timeout when probing |
| 200 ≠ honoured | Constraints can be silently ignored — validate output |
| Region footprint | 55 models in us-east-1, 33 in eu-central-1 |

## Next
- `02-migrating-from-openai.ipynb` — porting an existing OpenAI codebase
- `03-production-hardening-checklist.ipynb` — the pre-launch checklist